# vLLM Self-Hosting + Fine-Tuning on Databricks

End-to-end workflow for self-hosting a HuggingFace model on Databricks with vLLM:

1. Download model from HuggingFace and store weights in Unity Catalog
2. Serve with vLLM on classic GPU compute
3. Fine-tune with LoRA/PEFT
4. Re-serve the fine-tuned model with vLLM
5. Register to Unity Catalog as a governed model
6. Observe with MLflow tracing

**Requirements**: Databricks Runtime 15.4 LTS ML, classic GPU cluster (e.g., `g4dn.xlarge` or `g5.xlarge`)

## 1. Install Dependencies

In [ ]:
%pip install "vllm==0.6.3.post1" "transformers==4.46.0" "peft>=0.13.0" "trl>=0.9.0" "datasets>=2.14.0"
dbutils.library.restartPython()

## 2. Configuration

Update `HF_MODEL_NAME` with your model and the UC paths with your catalog/schema.

In [ ]:
import os
import time
import json
import shutil
import gc
from pathlib import Path
from typing import Optional, List

import mlflow
import mlflow.pyfunc
import pandas as pd
import torch
from databricks.sdk import WorkspaceClient

# --- Model Configuration ---
# Replace with your HuggingFace model (e.g., your OCR model repo ID)
HF_MODEL_NAME = "facebook/opt-125m"

# --- Unity Catalog Configuration ---
# Update these to your catalog/schema
UC_CATALOG = "mh_sandbox"
UC_SCHEMA = "vllm_poc"
UC_VOLUME = "model_weights"
UC_REGISTERED_MODEL = f"{UC_CATALOG}.{UC_SCHEMA}.opt125m_finetuned"

# --- Paths ---
LOCAL_CACHE = "/local_disk0/tmp/hf_model_cache"
UC_VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}"

# --- Inference defaults ---
DEFAULT_MAX_TOKENS = 128

os.makedirs(LOCAL_CACHE, exist_ok=True)
client = WorkspaceClient()
print(f"Workspace: {client.config.host}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Create UC Volume

Unity Catalog Volumes provide governed storage for model weights.

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}.{UC_VOLUME}")
print("UC catalog/schema/volume ready")

## 4. Download Model from HuggingFace and Store in UC

Model weights are stored as a governed object in Unity Catalog via Volumes.
This ensures access control, lineage, and auditability.

In [ ]:
from huggingface_hub import snapshot_download

def download_hf_model(model_name: str, cache_dir: str) -> str:
    """Download HF model to local cache. Returns local path."""
    cache_path = Path(cache_dir) / model_name.replace("/", "_")
    if cache_path.exists() and (cache_path / "config.json").exists():
        print(f"Already cached: {cache_path}")
        return str(cache_path)
    print(f"Downloading {model_name}...")
    cache_path.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id=model_name, local_dir=str(cache_path))
    print(f"Downloaded to: {cache_path}")
    return str(cache_path)

local_model_path = download_hf_model(HF_MODEL_NAME, LOCAL_CACHE)

### Copy weights to UC Volume

In [ ]:
uc_model_dir = f"{UC_VOLUME_PATH}/{HF_MODEL_NAME.replace('/', '_')}"
os.makedirs(uc_model_dir, exist_ok=True)

print(f"Copying model weights to UC Volume: {uc_model_dir}")
local_path = Path(local_model_path)
file_count = 0
for f in local_path.rglob("*"):
    if f.is_file():
        rel = f.relative_to(local_path)
        dest = Path(uc_model_dir) / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(str(f), str(dest))
        file_count += 1
print(f"Copied {file_count} files to UC Volume: {uc_model_dir}")

## 5. Serve with vLLM on Classic GPU

[vLLM](https://docs.vllm.ai/) provides high-throughput inference with PagedAttention,
continuous batching, and an OpenAI-compatible API. It runs on classic GPU compute
where you have full control over the serving infrastructure.

In [ ]:
from vllm import LLM, SamplingParams

print("Initializing vLLM engine...")
llm = LLM(
    model=local_model_path,
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.70,
    max_num_seqs=32,
    swap_space=0,
)
print("vLLM engine ready")

### Sanity test

In [ ]:
# OPT-125M is a base completion model (not instruction-tuned), so use a completion-style prompt
sp = SamplingParams(temperature=0.1, max_tokens=64)
test_prompts = [
    "The key benefits of machine learning are",
    "A large language model is a type of",
]
outputs = llm.generate(test_prompts, sampling_params=sp)
for i, out in enumerate(outputs):
    print(f"Prompt: {test_prompts[i]}")
    print(f"Completion: {out.outputs[0].text.strip()}\n")

## 6. Fine-Tune with LoRA (PEFT)

vLLM is an *inference* engine. Fine-tuning uses HuggingFace Transformers + [PEFT](https://huggingface.co/docs/peft).
After fine-tuning, we merge the LoRA adapter back into the base model and re-serve with vLLM.

### Free vLLM GPU memory before fine-tuning

In [ ]:
del llm
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Available: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

### Load base model and apply LoRA adapter

In [ ]:
from transformers import OPTForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = OPTForCausalLM.from_pretrained(HF_MODEL_NAME)
model.half().cuda()

# LoRA configuration — adjust target_modules for your model architecture
lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### 6a. Prepare Training Data

Replace this with your own dataset. The examples below demonstrate
document field extraction — swap in your domain-specific training pairs.

In [ ]:
from datasets import Dataset

train_data = [
    {"text": "Extract the key fields from this invoice: Invoice #1234, Date: 2024-01-15, Total: $5,432.10, Vendor: Acme Corp\nOutput: Invoice Number: 1234 | Date: 2024-01-15 | Total: $5,432.10 | Vendor: Acme Corp"},
    {"text": "Extract the key fields from this invoice: PO-9876, Date: 2024-03-22, Amount Due: $12,750.00, From: Global Systems Inc\nOutput: PO Number: 9876 | Date: 2024-03-22 | Amount Due: $12,750.00 | Vendor: Global Systems Inc"},
    {"text": "Extract the key fields from this invoice: Invoice #4455, Date: 2024-05-10, Total: $890.00, Vendor: CloudTech LLC\nOutput: Invoice Number: 4455 | Date: 2024-05-10 | Total: $890.00 | Vendor: CloudTech LLC"},
    {"text": "Extract the key fields from this invoice: Invoice #7721, Date: 2024-07-03, Total: $22,100.50, Vendor: DataPipe Inc\nOutput: Invoice Number: 7721 | Date: 2024-07-03 | Total: $22,100.50 | Vendor: DataPipe Inc"},
    {"text": "Extract the key fields from this invoice: PO-3344, Date: 2024-09-18, Amount Due: $6,750.00, From: Nexus Analytics\nOutput: PO Number: 3344 | Date: 2024-09-18 | Amount Due: $6,750.00 | Vendor: Nexus Analytics"},
    {"text": "Extract the key fields from this invoice: Invoice #5599, Date: 2024-02-28, Total: $1,200.00, Vendor: Summit Group\nOutput: Invoice Number: 5599 | Date: 2024-02-28 | Total: $1,200.00 | Vendor: Summit Group"},
    {"text": "Extract the key fields from this invoice: Invoice #8832, Date: 2024-11-05, Total: $45,000.00, Vendor: Alpine Services\nOutput: Invoice Number: 8832 | Date: 2024-11-05 | Total: $45,000.00 | Vendor: Alpine Services"},
    {"text": "Extract the key fields from this invoice: PO-6677, Date: 2024-04-12, Amount Due: $3,325.75, From: Vertex Solutions\nOutput: PO Number: 6677 | Date: 2024-04-12 | Amount Due: $3,325.75 | Vendor: Vertex Solutions"},
    {"text": "Extract the key fields from this invoice: Invoice #2210, Date: 2024-08-22, Total: $15,800.00, Vendor: Horizon Tech\nOutput: Invoice Number: 2210 | Date: 2024-08-22 | Total: $15,800.00 | Vendor: Horizon Tech"},
    {"text": "Extract the key fields from this invoice: Invoice #9901, Date: 2024-06-30, Total: $7,450.25, Vendor: Pacific Digital\nOutput: Invoice Number: 9901 | Date: 2024-06-30 | Total: $7,450.25 | Vendor: Pacific Digital"},
]

dataset = Dataset.from_list(train_data)
print(f"Training samples: {len(dataset)}")

### 6b. Run Fine-Tuning

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="/local_disk0/tmp/lora_output",
    num_train_epochs=20,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="no",
    warmup_ratio=0.05,
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=sft_config,
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("Fine-tuning complete")

### 6c. Merge LoRA Adapter and Save

Merge the LoRA weights back into the base model for clean vLLM serving.

In [ ]:
adapter_path = "/local_disk0/tmp/lora_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved: {adapter_path}")

del model, trainer
gc.collect()
torch.cuda.empty_cache()

print("Reloading base model for adapter merge...")
base_model = OPTForCausalLM.from_pretrained(HF_MODEL_NAME)
base_model.half().cuda()

merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_path = "/local_disk0/tmp/merged_model"
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f"Merged model saved: {merged_path}")

del base_model, merged_model
gc.collect()
torch.cuda.empty_cache()

## 7. Serve Fine-Tuned Model with vLLM

Reload the merged model into vLLM and verify inference.

In [ ]:
from vllm import LLM, SamplingParams

print("Loading fine-tuned model into vLLM...")
llm_ft = LLM(
    model=merged_path,
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.70,
    max_num_seqs=32,
    swap_space=0,
)

test_prompt = "Extract the key fields from this invoice: Invoice #5577, Date: 2024-08-10, Total: $8,200.00, Vendor: DataCo Analytics\nOutput:"
sp = SamplingParams(temperature=0.1, max_tokens=64, stop=["\n\n"])
out = llm_ft.generate([test_prompt], sampling_params=sp)
print("Fine-tuned model response:")
print(f"Prompt: {test_prompt}")
print(f"Output: {out[0].outputs[0].text.strip()}")

## 8. Register Fine-Tuned Model to Unity Catalog

Register the model as a governed object in Unity Catalog via MLflow.
This provides versioning, lineage, access controls, and a path to Model Serving deployment.

In [ ]:
del llm_ft
gc.collect()
torch.cuda.empty_cache()

In [ ]:
class VLLMPyfuncModel(mlflow.pyfunc.PythonModel):
    """MLflow pyfunc wrapper for vLLM inference at serving time."""

    def load_context(self, context):
        from vllm import LLM
        model_path = context.artifacts["model_dir"]
        self.llm = LLM(model=model_path, dtype="half", enforce_eager=True,
                        gpu_memory_utilization=0.70, max_num_seqs=32, swap_space=0)

    def predict(self, context, model_input):
        from vllm import SamplingParams
        prompts = model_input["prompt"].tolist()
        sp = SamplingParams(temperature=0.1, max_tokens=128)
        outputs = self.llm.generate(prompts, sampling_params=sp)
        return pd.DataFrame({
            "prompt": prompts,
            "response": [o.outputs[0].text.strip() for o in outputs],
        })

### Log and register model

In [ ]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

mlflow.set_registry_uri("databricks-uc")

signature = ModelSignature(
    inputs=Schema([ColSpec("string", "prompt")]),
    outputs=Schema([ColSpec("string", "prompt"), ColSpec("string", "response")]),
)

deploy_model = VLLMPyfuncModel()

with mlflow.start_run(run_name="vllm_finetuned_lora") as run:
    mlflow.log_params({
        "base_model": HF_MODEL_NAME,
        "inference_engine": "vllm",
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 20,
        "learning_rate": 3e-4,
    })

    mlflow.log_artifacts(merged_path, artifact_path="model_dir")

    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=deploy_model,
        signature=signature,
        artifacts={"model_dir": merged_path},
        pip_requirements=[
            "vllm==0.6.3.post1",
            "transformers==4.46.0",
            "torch",
            "mlflow>=2.14.1",
        ],
    )

    model_uri = f"runs:/{run.info.run_id}/model"
    registered = mlflow.register_model(model_uri=model_uri, name=UC_REGISTERED_MODEL)

print(f"Registered: {registered.name} v{registered.version}")
print(f"Model is now a governed object in Unity Catalog")

## 9. Observe with MLflow Tracing

Wrap vLLM inference calls with MLflow tracing for production observability.
Each call is logged with input, output, and latency in the MLflow Experiment UI.

In [ ]:
from vllm import LLM, SamplingParams

llm_ft = LLM(
    model=merged_path,
    dtype="half",
    enforce_eager=True,
    gpu_memory_utilization=0.70,
    max_num_seqs=32,
    swap_space=0,
)

In [ ]:
import mlflow

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/vllm_poc_tracing")

@mlflow.trace(name="vllm_inference")
def traced_inference(prompt: str) -> str:
    sp = SamplingParams(temperature=0.1, max_tokens=64, stop=["\n\n"])
    out = llm_ft.generate([prompt], sampling_params=sp)
    return out[0].outputs[0].text.strip()

test_prompts = [
    "Extract the key fields from this invoice: Invoice #9999, Date: 2024-12-01, Total: $3,100.00, Vendor: FinTech Solutions\nOutput:",
    "Extract the key fields from this invoice: PO-5500, Date: 2024-10-15, Amount Due: $18,900.00, From: Meridian Corp\nOutput:",
]

for p in test_prompts:
    result = traced_inference(p)
    print(f"Prompt: {p}")
    print(f"Output: {result}\n---")

print("Check the MLflow Experiment UI to see traces")

## 10. Deploy to Model Serving (Serverless)

Deploy the UC-registered model to a GPU-backed Model Serving endpoint.
This gives you autoscaling, scale-to-zero, an OpenAI-compatible REST API, and AI Gateway integration.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

w = WorkspaceClient()

ENDPOINT_NAME = "vllm-finetuned-poc"

# Get the latest version of the registered model
latest_version = max(
    w.model_versions.list(model_name=UC_REGISTERED_MODEL),
    key=lambda v: int(v.version),
).version

print(f"Deploying {UC_REGISTERED_MODEL} v{latest_version} to endpoint '{ENDPOINT_NAME}'...")

w.serving_endpoints.create_and_wait(
    name=ENDPOINT_NAME,
    config=EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=UC_REGISTERED_MODEL,
                entity_version=str(latest_version),
                workload_size="Small",
                workload_type="GPU_SMALL",
                scale_to_zero_enabled=True,
            )
        ]
    ),
)
print(f"Endpoint '{ENDPOINT_NAME}' is ready")

### Query the endpoint

In [ ]:
import requests

endpoint_url = f"{w.config.host}/serving-endpoints/{ENDPOINT_NAME}/invocations"
token = w.config.authenticate()

response = requests.post(
    endpoint_url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={
        "dataframe_records": [
            {"prompt": "Extract the key fields from this invoice: Invoice #7788, Date: 2024-11-20, Total: $4,500.00, Vendor: Apex Corp\nOutput:"}
        ]
    },
)
print(response.json())

## 11. Serverless vs Classic GPU — When to Use What

| | Serverless (Foundation Model APIs) | Classic GPU (vLLM) |
|---|---|---|
| **Custom HuggingFace models** | Not supported | Full control over model and infra |
| **Databricks-managed models** (Llama, DBRX, Mixtral) | Fully managed, pay-per-token | Not needed |
| **Fine-tuning** | Via Mosaic fine-tuning only | Any framework (PEFT, full fine-tune) |
| **vLLM configuration** | Not user-configurable | Full control |
| **Scale to zero** | Automatic | Configurable on Model Serving |
| **Best for** | Standard LLM chat, embeddings | Custom/specialized models (OCR, vision, domain-specific) |

**For custom HuggingFace models**: Classic GPU compute with vLLM is the recommended path.
Register the fine-tuned model to UC (done above), then deploy to a
[Model Serving endpoint](https://docs.databricks.com/en/machine-learning/model-serving/index.html) with GPU compute for production.

## 12. Cost Estimation — Classic GPU Compute (AWS)

All-Purpose Compute pricing (Databricks Premium plan, AWS, DBU + EC2 combined):

| Instance | GPU | VRAM | DBU/hr | Total $/hr | Good for |
|---|---|---|---|---|---|
| `g4dn.xlarge` | 1x T4 | 16 GB | 0.71 | $0.92 | Small models, dev/POC |
| `g5.xlarge` | 1x A10G | 24 GB | 1.36 | $1.75 | Models up to 7B, dev/test |
| `g5.2xlarge` | 1x A10G | 24 GB | 1.64 | $2.11 | 7B models, light production |
| `g5.12xlarge` | 4x A10G | 96 GB | 7.69 | $9.90 | 13-30B models, tensor parallel |

Jobs Compute is cheaper ($0.15/DBU vs $0.55/DBU for All-Purpose).

**Recommendations by model size:**
- **1-3B params** (e.g., OCR models): `g4dn.xlarge` ($0.92/hr) or `g5.xlarge` ($1.75/hr)
- **7B params**: `g5.2xlarge` ($2.11/hr) with scale-to-zero
- **13B+ params**: `g5.12xlarge` ($9.90/hr) with tensor parallelism

*Sources: [Databricks Pricing Calculator](https://www.databricks.com/product/pricing/product-pricing/instance-types) (Premium, AWS, GPU Instances). EC2 on-demand rates from AWS. Negotiated enterprise pricing may differ.*

## Summary and Next Steps

**What this notebook demonstrated:**
1. Download a HuggingFace model and store weights in a UC Volume (governed storage)
2. Serve with vLLM on classic GPU compute
3. Fine-tune with LoRA/PEFT, merge the adapter
4. Re-serve the fine-tuned model with vLLM
5. Register to Unity Catalog as a versioned, governed model
6. Observe inference with MLflow tracing
7. Deploy to a Model Serving endpoint (serverless, autoscaling, scale-to-zero)
8. Serverless vs classic GPU comparison and cost estimates

**To adapt for your model:**
1. Set `HF_MODEL_NAME` to your model's HuggingFace repo ID
2. Prepare your training dataset (e.g., OCR image-to-text pairs)
3. Update LoRA `target_modules` to match your model's architecture
4. For vision/multimodal models, use the appropriate Transformers pipeline
5. Deploy to a [Model Serving endpoint](https://docs.databricks.com/en/machine-learning/model-serving/index.html) for production traffic